In [17]:
from help_functions import read_real_graph, read_list
import numpy as np 
from pred import convertToPermHungarian2new
import pandas as pd

In [18]:
def printR(name,dataset,accuracy):
    print('---- ',name, '----')
    print('---- ',dataset, '----')
    print('----> Accuracy:', accuracy)
    print() 

Read Graphs: Give the path to read the files. You should give the number of nodes as input also.

In [19]:
dataset="douban"
graphsizeA=3906
graphsizeB=1118
G = read_real_graph(n =graphsizeA , name_ = f'./Data/data/douban/douban_t_edge.txt')
G_Q = read_real_graph(n = graphsizeB, name_ = f'./Data/data/douban/douban_s_edge.txt')
F2 = pd.read_csv(f"./Data/Full-dataset/attribute/doubanattr1.csv", header=None).iloc[:, 1:].to_numpy()
F1 = pd.read_csv(f"./Data/Full-dataset/attribute/doubanattr2.csv", header=None).iloc[:, 1:].to_numpy()

Making ./Data/data/douban/douban_t_edge.txt graph...
Done ./Data/data/douban/douban_t_edge.txt ...
Making ./Data/data/douban/douban_s_edge.txt graph...
Done ./Data/data/douban/douban_s_edge.txt ...


Read Ground truth.

In [25]:
pairs = []
gtsize=1118
with open(f'./Data/data/douban/douban_ground_True.txt', "r") as f:
    for line in f:
        a, b = line.strip().split()
        pairs.append((int(a), int(b)))
max_A = graphsizeB
#max_B = 1118
if dataset == "allmv_tmdb":
    max_A += 1

A_to_B = [-1] * max_A
B_to_A = [-1] * max_A

for a, b in pairs:
    if 0 <= a < max_A:
        A_to_B[a] = b
    if 0 <= b < max_A:
        B_to_A[b] = a

import and run the algorithm

In [21]:
from pred import AlpineL

mun=0.1
_, list_of_nodes, _ = AlpineL(G_Q.copy(), G.copy(),F1,F2,mun,weight=2)


Evaluate the algorithm.

In [27]:
accuracy = np.sum(np.array(A_to_B)==np.array(list_of_nodes))/gtsize
accuracy = np.sum(np.array(B_to_A)==np.array(list_of_nodes))/gtsize

printR("Alpine",dataset,accuracy)  


----  Alpine ----
----  douban ----
----> Accuracy: 0.6994633273703041



For the seeded version, load also the seeds/anchors. We will keep the same dataset for the example. We set the ground truth ratio to 10

In [38]:
from superviseutil import synchronize_features,seed_link

ratio=0.1
iter=1
data_GT = np.loadtxt(f"./Data/data/douban/douban_ground_True_{ratio}_{iter}.txt", dtype=int)
anchors_G = data_GT[:, 0].tolist()
anchors_GQ = data_GT[:, 1].tolist()
F2_n,F1_n=synchronize_features(F2,F1,anchors_G,anchors_GQ)
G1,G1_Q=seed_link(anchors_G,anchors_GQ,G,G_Q)

Add seed links : 0	

Import and run the algorithm

In [40]:
from pred import Alpine_supervised
_, list_of_nodes, _ = Alpine_supervised(G1_Q.copy(), G1.copy(),F1_n,F2_n,anchors_GQ,anchors_G,mun,weight=2)


Evaluate the algorithm

In [41]:
accuracy = np.sum(np.array(A_to_B)==np.array(list_of_nodes))/gtsize
accuracy = np.sum(np.array(B_to_A)==np.array(list_of_nodes))/gtsize

printR("Alpine",dataset,accuracy)  

----  Alpine ----
----  douban ----
----> Accuracy: 0.7674418604651163

